## 議員データ収集

In [ ]:
import bs4
import requests
import re
from urllib.parse import urljoin
import os
from params.paths import ROOT_DIR
import pandas as pd
import time
from tqdm import tqdm
import json

from file_handling.file_read_writer import write_json, read_json
from collections import Counter
from api_requests.prompter import DeepResearchGemini

SHUGIIN_REPR_URL = 'https://kokkai.sugawarataku.net/giin/rgiin.html'
SANGIIN_REPR_URL = 'https://kokkai.sugawarataku.net/giin/cgiin.html'

LOWER_HOUSE_DATA_DIR = os.path.join(ROOT_DIR, 'data', 'data_shugiin', 'repr_list')
LOWER_HOUSE_DATA_HISTORICAL_TMP = os.path.join(LOWER_HOUSE_DATA_DIR, 'historical')
UPPER_HOUSE_DATA_DIR = os.path.join(ROOT_DIR, 'data', 'data_sangiin', 'repr_list')
UPPER_HOUSE_DATA_HISTORICAL_TMP = os.path.join(UPPER_HOUSE_DATA_DIR, 'historical')
LOWER_HOUSE_LOG = os.path.join(LOWER_HOUSE_DATA_DIR, 'log.txt')
UPPER_HOUSE_LOG = os.path.join(UPPER_HOUSE_DATA_DIR, 'log.txt')
os.makedirs(LOWER_HOUSE_DATA_HISTORICAL_TMP, exist_ok=True)
os.makedirs(UPPER_HOUSE_DATA_HISTORICAL_TMP, exist_ok=True)

In [ ]:
def get_repr_data_prompt(name):
	prompt =  """\
			この政治家のデータを集めてほしいです。
			政治家の名前はNAMEです。
			返答フォーマットは以下のようにしてください。
			
			{
			"name_kanji": "名前",
			"name_kana": "名前のふりがな",
			"years": [
			"当選年-当選月-当選日",
			"当選年-当選月-当選日",
			...
			],
			"election_data": [
				{
					"year": "当選年",
					"month": "当選月",
					"day": "当選日",
					"election_name": "当選回次",
					"district": "選挙区",
					"party": "政党",
					"result": "当選",
					"election_freq": "（1回目）"
				},
				...
			]
			}

			返答フォーマットの例：
			{
				"name_kanji": "七条明",
				"name_kana": "しちじょうあきら",
				"years": [
					"1993-07-18",
					...
				],
				"election_data": [
					{
						"year": "1993年",
						"month": "7月",
						"day": "18日",
						"election_name": "第40回衆議院議員総選挙",
						"district": "徳島全県区",
						"party": "自由民主党",
						"result": "当選",
						"election_freq": "（1回目）"
					},
					...
				]
			}
	"""
	prompt = prompt.replace("NAME", name)

	return prompt

research_system_prompt = """\
	あなたは政治家のデータを集めるアシスタントです。常に最新のデータを集めるために必ず検索をすることを心がけてください。
	また、同姓同名の政治家にも気を付けてください。もし同姓同名政治家がいた場合、これから与える返答フォーマットは無視してください。
	その際は「同姓同名の政治家がいる可能性があります」と返答してください。ちなみに地方政治家は含めず、国会議員のみを集めてください。
	jsonを回答する際には余計な説明等をつけず、生のjsonのみを返答してください。コードブロック（```json）をつけることは絶対にしないでください。
	あなたの返答フォーマットはそのままjson.loadsでパースできるようにしてください。
	"""


In [ ]:
class ReprHistoricalDataCollector:
	def __init__(self):
		self.prompter = DeepResearchGemini(model_name="gemini-2.5-pro")

	def process_one_election_row(self, row):

		def try_with_backup(cls1, cls2):
			attempt1 = row.find('div', {'class': cls1})
			if attempt1:
				return attempt1.text
			attempt2 = row.find('div', {'class': cls2})
			if attempt2:
				return attempt2.text
			return ""
			
		try:
			
			year = row.find('div', {'class': 'el1'}).text
			month = row.find('div', {'class': 'el2'}).text
			day = row.find('div', {'class': 'el3'}).text
			election_name = row.find('div', {'class': 'el4'}).text
			district = try_with_backup('el5', 'elc5')
			party = try_with_backup('el6', 'elc6')
			result = try_with_backup('el7', 'elc7')
			election_freq = row.find('div', {'class': 'el8'}).text
		except Exception as e:
			print(f'Error processing row: {row},\n error: {e}')
			raise e
		return {'year': year, 'month': month, 'day': day, 'election_name': election_name, 'district': district, 'party': party, 'result': result, 'election_freq': election_freq}

	def retrieve_info_of_one_repr(self, url):
		html = requests.get(url).content
		soup = bs4.BeautifulSoup(html, 'html.parser')
		repr_data = soup.find_all('div', {'class':'jt2'})
		name_kanji = repr_data[0].text
		name_kana = repr_data[1].text
		years = re.findall(r"\d{4}/\d{2}/\d{2}", repr_data[3].text)
		years = [year.replace('/', '-') for year in years]

		election_data = soup.find_all('div', {'class':'em1'})
		election_data = [self.process_one_election_row(row) for row in election_data]

		return {'name_kanji': name_kanji, 'name_kana': name_kana, 'years': years, 'election_data': election_data}

	def get_mismatched_names(self, house:str):
		if house == 'upper':
			url = SANGIIN_REPR_URL
		elif house == 'lower':
			url = SHUGIIN_REPR_URL
		resp = requests.get(url)
		resp.encoding = "cp932"
		soup = bs4.BeautifulSoup(resp.text, 'lxml')
		links = soup.find_all('span', {'class':'zt5'})
		names = [link.find('a').get_text(strip=True) for link in links]
		hrefs = [link.find('a').get('href') for link in links]

		mismatched_names = []
		for idx, (name, href) in enumerate(zip(names, hrefs)):
			repr_link = urljoin(url, href)
			repr_data = self.retrieve_info_of_one_repr(repr_link)
			if repr_data["name_kanji"] != name:
				mismatched_names.append((name, repr_data["name_kanji"]))
				print(f'Name mismatch: {name} vs {repr_data["name_kanji"]}')
		return mismatched_names


	def collect(self, house:str):
		if house == 'upper':
			url = SANGIIN_REPR_URL
			log_path = UPPER_HOUSE_LOG
		elif house == 'lower':
			url = SHUGIIN_REPR_URL
			log_path = LOWER_HOUSE_LOG
		tempDir = UPPER_HOUSE_DATA_HISTORICAL_TMP if house == 'upper' else LOWER_HOUSE_DATA_HISTORICAL_TMP
		houseDir = UPPER_HOUSE_DATA_DIR if house == 'upper' else LOWER_HOUSE_DATA_DIR
		print(f'Collecting historical data for {house} house representatives into {tempDir}')
		resp = requests.get(url)
		resp.encoding = "cp932"
		soup = bs4.BeautifulSoup(resp.text, 'lxml')
		links = soup.find_all('span', {'class':'zt5'})
		print(links)
		names = [link.find('a').get_text(strip=True) for link in links]
		print(names)
		hrefs = [link.find('a').get('href') for link in links]
		if Counter(names).most_common()[0][1] > 1:
			print(Counter(names).most_common())
			raise ValueError('There are duplicate names in the historical data')

		if os.path.exists(log_path):
			with open(log_path, 'r') as f:
				done_names = f.readlines()
		else:
			done_names = []


		for idx, (name, href) in enumerate(zip(names, hrefs)):
			if name in done_names:
				print(f'{name} already done')
				continue
			print(f'Processing {name}-{idx/len(names)*100:.2f}%')
			repr_path = os.path.join(tempDir, f'{name}.json')
			user_input = None
			if os.path.exists(repr_path):
				repr_data = read_json(repr_path)
				if repr_data["name_kana"] != "":
					print(f'{name} already exists')
					with open(log_path, 'a') as f:
						f.write(f'{name}\n')
					continue
				else:
					print(f"Data incomplete for {name}")
					prompt = get_repr_data_prompt(name)
					count = 0
					while True:
						try:
							reply, _, _ = self.prompter.prompt(prompt, research_system_prompt)
							print("REPLY:", reply)
							if reply == "同姓同名の政治家がいる可能性があります":
								# user_input = input(f"同姓同名の政治家がいる可能性があります。{name}の生年月日を入力してください。そっちをまず収集します。")
								user_input = "skip"
								print("SKIPPING")
								break
								
								# reply, _, _ = self.prompter.prompt(
								# 	prompt + "\n" + user_input+"が生年月日の政治家のほうの情報を収集してください。", research_system_prompt
								# )
								# print("REPLY:", reply)
							repr_data = json.loads(reply)
							break
						except Exception as e:
							print(f"Error parsing reply: {e}")
							time.sleep(1)
				if user_input == "skip":
					continue
			else:
				repr_link = urljoin(url, href)
				repr_data = self.retrieve_info_of_one_repr(repr_link)
				
				if repr_data["name_kanji"] != name:
					print(f'{name} has a different name in the historical data {repr_data["name_kanji"]}')
					count = 0
					while True:
						try:
							prompt = get_repr_data_prompt(name)
							reply, _, _ = self.prompter.prompt(prompt, research_system_prompt)
							if reply == "同姓同名の政治家がいる可能性があります":
								user_input = input(f"同姓同名の政治家がいる可能性があります。{name}の生年月日を入力してください。そっちをまず収集します。")
								reply, _, _ = self.prompter.prompt(
									prompt + "\n" + user_input+"が生年月日の政治家のほうの情報を収集してください。", research_system_prompt
								)
							print("REPLY:", reply)
							repr_data = json.loads(reply)
							break
						except Exception as e:
							print(f"Error parsing reply: {e}")
							time.sleep(1)
							count += 1
							if count > 3:
								raise e
					repr_data = json.loads(reply)
			path = os.path.join(tempDir, f'{name}{user_input if user_input else ""}.json')
			write_json(repr_data, path)
			with open(log_path, 'a') as f:
				f.write(f'{name}\n')
			time.sleep(1)

		all_repr_data = []
		if Counter(list(os.listdir(tempDir))).most_common()[0][1] > 1:
			print(Counter(list(os.listdir(tempDir))).most_common())
			raise ValueError('There are duplicate names in the historical data')

		covered_files = set()
		for repr_file in os.listdir(tempDir):
			if repr_file in covered_files:
				raise ValueError(f'{repr_file} already exists')
			covered_files.add(repr_file)
			repr_path = os.path.join(tempDir, repr_file)
			repr_data = read_json(repr_path)
			all_repr_data.append(repr_data)
		all_repr_data_names = [repr_data['name_kanji'] for repr_data in all_repr_data]
		if Counter(all_repr_data_names).most_common()[0][1] > 1:
			print(Counter(all_repr_data_names).most_common())
			raise ValueError('There are duplicate names in the historical data')

		write_json({'data':all_repr_data}, os.path.join(houseDir, 'historical.json'))

In [ ]:
sc = ReprHistoricalDataCollector()
sc.get_mismatched_names("lower")

In [ ]:
sc = ReprHistoricalDataCollector()
sc.collect("lower")
sc.collect("upper")

In [ ]:
LOWER_HOUSE_DATA_DIR = os.path.join(ROOT_DIR, 'data', 'data_shugiin', 'repr_list')
LOWER_HOUSE_TARGET_DIR = os.path.join(LOWER_HOUSE_DATA_DIR, 'current')
UPPER_HOUSE_DATA_DIR = os.path.join(ROOT_DIR, 'data', 'data_sangiin', 'repr_list')
UPPER_HOUSE_TARGET_DIR = os.path.join(UPPER_HOUSE_DATA_DIR, 'current')
LOWER_HOUSE_SOURCE_PATH = os.path.join(LOWER_HOUSE_DATA_DIR, "20250911_repr_list.json")
UPPER_HOUSE_SOURCE_PATH = os.path.join(UPPER_HOUSE_DATA_DIR, "20250911_repr_list.json")

current_repr_list_upper = read_json(UPPER_HOUSE_SOURCE_PATH)['reprs']
current_repr_list_lower = read_json(LOWER_HOUSE_SOURCE_PATH)['reprs']
print(current_repr_list_upper)
print(current_repr_list_lower)

os.makedirs(LOWER_HOUSE_TARGET_DIR, exist_ok=True)
os.makedirs(UPPER_HOUSE_TARGET_DIR, exist_ok=True)



## Now do the same for the current repr list

In [ ]:
from dotenv import load_dotenv
import psycopg2
from collections import Counter

load_dotenv()


def get_all_politicians_from_db(cur: psycopg2.extensions.cursor):
    cur.execute("SELECT name_kanji FROM person")
    return [row[0] for row in cur.fetchall()]



conn = None
try:
    conn = psycopg2.connect(
        dbname="kokkaidoc",
        user="postgres",
        password=os.getenv("PSQL_DATABASE_PASSWORD"),
        host="localhost",
        port="5432",
    )
    print("Connected.")

    with conn.cursor() as cur:
        politicians_in_db = get_all_politicians_from_db(cur)
        print(Counter(politicians_in_db).most_common())
        print('Got', len(politicians_in_db), 'politicians from db')
        
        politicians_in_db = set(politicians_in_db)

    print("Done.")

except Exception as e:
    if conn:
        conn.rollback()
    raise
finally:
    if conn:
        conn.close()

In [ ]:
from utils.string_process import clean_repr_name

def get_repr_data_prompt(name):
	prompt =  """\
			この政治家のデータを集めてほしいです。
			政治家の名前はNAMEです。
			返答フォーマットは以下のようにしてください。
			
			{
			"name_kanji": "名前",
			"name_kana": "名前のふりがな",
			"years": [
			"当選年-当選月-当選日",
			"当選年-当選月-当選日",
			...
			],
			"election_data": [
				{
					"year": "当選年",
					"month": "当選月",
					"day": "当選日",
					"election_name": "当選回次",
					"district": "選挙区",
					"party": "政党",
					"result": "当選",
					"election_freq": "（1回目）"
				},
				...
			]
			}

			返答フォーマットの例：
			{
				"name_kanji": "七条明",
				"name_kana": "しちじょうあきら",
				"years": [
					"1993-07-18",
					...
				],
				"election_data": [
					{
						"year": "1993年",
						"month": "7月",
						"day": "18日",
						"election_name": "第40回衆議院議員総選挙",
						"district": "徳島全県区",
						"party": "自由民主党",
						"result": "当選",
						"election_freq": "（1回目）"
					},
					...
				]
			}
	"""
	prompt = prompt.replace("NAME", name)

	return prompt

research_system_prompt = """\
	あなたは政治家のデータを集めるアシスタントです。常に最新のデータを集めるために必ず検索をすることを心がけてください。
	また、同姓同名の政治家にも気を付けてください。もし同姓同名政治家がいた場合、これから与える返答フォーマットは無視してください。
	その際は「同姓同名の政治家がいる可能性があります」と返答してください。ちなみに地方政治家は含めず、国会議員のみを集めてください。
	jsonを回答する際には余計な説明等をつけず、生のjsonのみを返答してください。コードブロック（```json）をつけることは絶対にしないでください。
	あなたの返答フォーマットはそのままjson.loadsでパースできるようにしてください。
	"""

def iterate_current_repr_list(repr_list):
	for party, reprs in repr_list.items():
		for repr in reprs:
			yield repr



In [ ]:
from api_requests.prompter import DeepResearchGemini
prompter = DeepResearchGemini(model_name="gemini-2.5-pro")
from file_handling.file_read_writer import write_json

for repr_list, target_dir in zip([current_repr_list_upper, current_repr_list_lower], [UPPER_HOUSE_TARGET_DIR, LOWER_HOUSE_TARGET_DIR]):
	for repr in iterate_current_repr_list(repr_list):
		name = clean_repr_name(repr['name'])
		if name in politicians_in_db:
			# skipping repr names that already exist in db
			continue
		if os.path.exists(os.path.join(target_dir, f'{name}.json')):
			# skipping repr names that already exist in target dir
			continue
		count = 0
		while True:
			try:
				prompt = get_repr_data_prompt(name)
				reply, _, _ = prompter.prompt(prompt, research_system_prompt+"\n必ず最新のデータを集めてください。ALWAYS SEARCH ON GOOGLE.")
				print("GOT REPLY")
				print(reply)
				reply = reply.replace("```json", "").replace("```", "")
				repr_data = json.loads(reply)
				write_json(repr_data, os.path.join(target_dir, f'{name}.json'))
				break
			except Exception as e:
				print(f"Error processing {name}: {e}")
				time.sleep(1)
				count += 1
				if count > 3:
					raise e
			continue
		
		


## Some representatives still slipped through

In [ ]:
missed_reprs = [
	'おおつつとむ',
	'うえのほたる',
	'むらかみもとのぶ',
	'にしかわまさと',
	'てらたまなぶ',
	'わかばやしけんた',
	'いはらたくみ',
	'えとうせいしろう',
	'てらたまなぶ',
	'かねだかつとし',
	'たかはしちづこ',
	'よしいえひろゆき',
	'かさいあきら',
	'まえがわきよしげ',
	'みやもとたけし',
	'とくながひさし',
	'たかがいえみこ'
]
target_dir = os.path.join(ROOT_DIR, 'data', 'tmp', 'missed_reprs')
prompter = DeepResearchGemini(model_name="gemini-2.5-pro")
os.makedirs(target_dir, exist_ok=True)
for name_hiragana in missed_reprs:
	prompt = get_repr_data_prompt(name_hiragana)
	count = 0
	while True:
		try:
			reply, _, _ = prompter.prompt(prompt, research_system_prompt+"\n必ず最新のデータを集めてください。ALWAYS SEARCH ON GOOGLE.")
			print("GOT REPLY")
			print(reply)
			reply = reply.replace("```json", "").replace("```", "")
			repr_data = json.loads(reply)
			write_json(repr_data, os.path.join(target_dir, f'{name_hiragana}.json'))
			break
		except Exception as e:
			print(f"Error processing {name_hiragana}: {e}")
			time.sleep(1)
			count += 1
			if count > 3:
				raise e
		continue

## Some cleaning of database and getting rid of duplicates

In [ ]:
from dotenv import load_dotenv
import shutil
import re
import os
load_dotenv()
from dbio.representative_db import connect_db, iterate_all_persons, get_election_result_by_person_id, get_x_account_by_person_id


conn = connect_db(
		dbname="kokkaidoc",
		user="postgres",
		password=os.getenv("PSQL_DATABASE_PASSWORD"),
		host="localhost",
		port="5432"
	)

# get rid of all white space from name_kana fields
with conn.cursor() as cur:
	for person in iterate_all_persons(cur):
		name_hiragana = person.name_kana
		name_hiragana_clean = re.sub(r"\s+", "", name_hiragana)
		if name_hiragana != name_hiragana_clean:
			cur.execute(
				"UPDATE person SET name_kana = %s WHERE person_id = %s",
				(name_hiragana_clean, person.person_id)
			)
			print(f'Updated {person.name_kanji} from "{name_hiragana}" to "{name_hiragana_clean}"')
	conn.commit()

In [ ]:
# make a list of all duplicate names in hiragana field
persons_with_duplicate_names = []
with conn.cursor() as cur:
	cur.execute(
		"SELECT name_kana, COUNT(*) FROM person GROUP BY name_kana HAVING COUNT(*) > 1"
	)
	rows = cur.fetchall()
	persons_with_duplicate_names = [row[0] for row in rows]
	print(f'Found {len(persons_with_duplicate_names)} duplicate names in hiragana')

print('Persons with duplicate names in hiragana:', persons_with_duplicate_names)


# for each duplicate name, print out their kanji names and election results
with conn.cursor() as cur:
	skip_until ='さとうしずお'
	for idx, name_kana in enumerate(persons_with_duplicate_names):
		if skip_until:
			if name_kana != skip_until:
				continue
			else:
				skip_until = None
		print(f'Processing duplicate name {idx+1}/{len(persons_with_duplicate_names)}: "{name_kana}"')
		cur.execute(
			"SELECT person_id, name_kanji FROM person WHERE name_kana = %s",
			(name_kana,)
		)
		rows = cur.fetchall()
		for row in rows:
			person_id = row[0]
			name_kanji = row[1]
			print(f'  person_id: {person_id}, name_kanji: {name_kanji}')
			election_results = get_election_result_by_person_id(cur, person_id)
			x_account = get_x_account_by_person_id(cur, person_id)
			print(f'    X account: {x_account.account_id if x_account else "None"}')
			print(f'    Election results:')
			for er in election_results:
				print(er)

		keep_id = input(f'Enter the person_id to keep for name_kana "{name_kana}": ')
		if not keep_id:
			continue
		keep_id = int(keep_id)
		if keep_id not in [row[0] for row in rows]:
			print(f'Invalid person_id {keep_id} for name_kana "{name_kana}", skipping deletion')
			continue
		# delete all other persons with this name_kana
		for row in rows:
			person_id = row[0]
			if person_id == keep_id:
				continue
			cur.execute(
				"DELETE FROM election_result WHERE person_id = %s",
				(person_id,)
			)
			print(f'Deleted election results for person_id {person_id}')
			cur.execute(
				"DELETE FROM x_account WHERE person_id = %s",
				(person_id,)
			)
			# delete this person
			cur.execute(
				"DELETE FROM person WHERE person_id = %s",
				(person_id,)
			)
			print(f'Deleted person_id {person_id}')
		conn.commit()

In [ ]:
merge_history = ['田名部匡省','岡田広']